# Lab 3B — A RAG Agent Whose Every Step Is Auditable

[Lab 2B](#) ended on a question: *if you cannot tell good retrieval from bad by
reading the answer, how would you ever know?*

The answer is that you stop reading answers and start reading **traces** — a record
of what each step received, produced and decided.

| Cell group | What you do |
|---|---|
| 0–2 | Point MLflow at **Unity Catalog**, and check that it actually landed there. |
| 3–4 | Build a RAG agent with one span per step. |
| 5 | Run it. |
| 6 | Read the trace back — including the exact chunks retrieved. |
| 7 | **Query your traces in SQL.** |

**Compute:** Serverless. **Prerequisite:** [Lab 3A](#) — you need the index.

In [0]:
%pip install -q "mlflow[databricks]" databricks-ai-search openai
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 1. Send traces to Unity Catalog, not the legacy store

This is the cell people get wrong, and the failure is silent.

Calling `mlflow.set_experiment(name)` on its own **works** — and quietly writes
traces to legacy workspace storage. You find out weeks later when you go looking for
the Delta tables and there are none.

Three things are required:

| | |
|---|---|
| `trace_location=UnityCatalog(...)` | the destination |
| `MLFLOW_TRACING_SQL_WAREHOUSE_ID` | a warehouse to provision and query the tables |
| An absolute experiment path | `/Users/you/name`. A bare name is rejected. |

In [0]:
import os, mlflow
from databricks.sdk import WorkspaceClient
from mlflow.entities.trace_location import UnityCatalog

CATALOG, SCHEMA = "agents_labs", "retail"
INDEX = f"{CATALOG}.{SCHEMA}.support_chunks_idx"
ENDPOINT = "agents-labs-vs"

w = WorkspaceClient()
user = w.current_user.me().user_name

# Find a warehouse rather than hard-coding an id.
whs = [x for x in w.warehouses.list()]
if not whs:
    raise RuntimeError("no SQL warehouse available — UC trace storage needs one")
os.environ["MLFLOW_TRACING_SQL_WAREHOUSE_ID"] = whs[0].id
print(f"  warehouse : {whs[0].name}  ({whs[0].id})")

EXPERIMENT = f"/Users/{user}/agents-labs-3b"
mlflow.set_experiment(
    experiment_name=EXPERIMENT,
    trace_location=UnityCatalog(catalog_name=CATALOG, schema_name=SCHEMA),
)
print(f"  experiment: {EXPERIMENT}")

  warehouse : Serverless Starter Warehouse  (c8729519c456cb8e)
  experiment: /Users/admin@datacouchoutlook.onmicrosoft.com/agents-labs-3b


### Verify it, do not assume it

A trace appearing in the MLflow UI proves the export worked. It does **not** prove
it went to Unity Catalog.

In [0]:
exp = mlflow.get_experiment_by_name(EXPERIMENT)
loc = exp.trace_location

print(f"  trace_location type : {type(loc).__name__}")
print(f"  location            : {loc}")
print(f"\n  Unity Catalog? {isinstance(loc, UnityCatalog)}")

assert isinstance(loc, UnityCatalog), "traces are going to legacy storage"
print("  ok — traces will land in Delta tables you can query")

  trace_location type : UnityCatalog
  location            : UnityCatalog(catalog_name='agents_labs', schema_name='retail', table_prefix='2155530130638727')

  Unity Catalog? True
  ok — traces will land in Delta tables you can query


## 2. Build the agent, one span per step

`@mlflow.trace` wraps a function in a span. The `span_type` is not decoration —
`RETRIEVER` is what makes MLflow record the retrieved documents in a form the
evaluation scorers in [Session 6](#) can read.

| Function | span_type | What the span records |
|---|---|---|
| `search_policy` | `RETRIEVER` | the query, the filter, every chunk **with its score** |
| `get_order` | `TOOL` | the arguments and the row returned |
| `answer` | `AGENT` | the question in, the answer out |

In [0]:
import json, warnings
from openai import OpenAI
from databricks.ai_search.client import VectorSearchClient

warnings.filterwarnings("ignore", message=".*serializer warnings.*")

_tok = w.config.authenticate()["Authorization"].removeprefix("Bearer ")
client = OpenAI(api_key=_tok, base_url=f"{w.config.host}/serving-endpoints")
_ix = VectorSearchClient(workspace_url=w.config.host, personal_access_token=_tok,
                         disable_notice=True).get_index(endpoint_name=ENDPOINT,
                                                        index_name=INDEX)
MODEL = "databricks-claude-sonnet-5"

SYSTEM = """You are a support agent for an office furniture retailer.

Answer only from the policy excerpts you retrieve and the order facts you look
up. If the excerpts do not cover the question, say so rather than guessing.
Cite the document id of every excerpt you rely on, like [DOC-003]."""


@mlflow.trace(span_type="RETRIEVER")
def search_policy(query: str, k: int = 3, audience: str = "customer"):
    r = _ix.similarity_search(query_text=query,
                              columns=["chunk_id", "doc_id", "title", "chunk"],
                              filters={"audience": audience}, num_results=k)
    return [{"chunk_id": x[0], "doc_id": x[1], "title": x[2],
             "text": x[3], "score": x[-1]}
            for x in (r.get("result", {}).get("data_array", []) or [])]


@mlflow.trace(span_type="TOOL")
def get_order(order_id: str):
    rows = spark.sql(f"""
        SELECT o.order_id, o.status, o.item, o.units, o.revenue, o.order_date,
               c.name AS customer, c.region, c.tier
        FROM {CATALOG}.{SCHEMA}.orders o
        JOIN {CATALOG}.{SCHEMA}.customers c USING (customer_id)
        WHERE o.order_id = '{order_id.replace("'", "")}'
    """).collect()
    return rows[0].asDict() if rows else {"error": f"no such order: {order_id}"}


TOOLS = [
    {"type": "function", "function": {
        "name": "search_policy",
        "description": "Search company policy: delivery, returns, warranty, billing.",
        "parameters": {"type": "object",
                       "properties": {"query": {"type": "string"}},
                       "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "get_order",
        "description": "Look up one order and the customer who placed it.",
        "parameters": {"type": "object",
                       "properties": {"order_id": {"type": "string"}},
                       "required": ["order_id"]}}},
]
IMPL = {"search_policy": lambda a: search_policy(a["query"]),
        "get_order": lambda a: get_order(a["order_id"])}

print("  spans defined: search_policy (RETRIEVER), get_order (TOOL)")

  spans defined: search_policy (RETRIEVER), get_order (TOOL)


## 3. The loop — one more span around the whole turn

In [0]:
def _text(m):
    c = m.content
    if c is None: return ""
    if isinstance(c, str): return c
    return "\n".join(b.get("text", "") for b in c
                     if isinstance(b, dict) and b.get("type") == "text").strip()


@mlflow.trace(span_type="AGENT")
def answer(question: str, max_steps: int = 6) -> str:
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        msg = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS, max_tokens=800
        ).choices[0].message
        messages.append(msg.model_dump(exclude_none=True))
        if not msg.tool_calls:
            return _text(msg)
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            print(f"  step {step}: {tc.function.name}({args})")
            out = IMPL[tc.function.name](args)
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(out, default=str)[:6000]})
    return "(no answer within max_steps)"


print("  agent ready")

  agent ready


## 4. Run it

The same question as [Lab 2A](#) — so you can compare the answer *and* now inspect
how it was produced.

In [0]:
Q = "My order ORD-1044 hasn't arrived. How long should it take, and what are my options?"
print(answer(Q))

  step 1: get_order({'order_id': 'ORD-1044'})
  step 1: search_policy({'query': 'delivery time and late delivery options'})
[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Good news — according to our system, order **ORD-1044** (5 standing desks, ordered 2026-09-18, for Helsinki Works) is actually already marked as **delivered**.

For reference on timelines: standard delivery to the Nordics normally takes **5–7 working days**, since shipments are consolidated at our Hamburg hub before onward transport [DOC-003]. Express delivery (extra 18 GBP) can halve that timeframe [DOC-003].

Since the system shows it as delivered, a few options if you haven't actually received it:
1. Double-check with anyone else at your location (reception, mailroom, neighboring offices) in case it was signed for by someone else.
2. Check for a del

Trace(trace_id=trace:/agents_labs.retail.2155530130638727/446d61e78b5bc1df8413c49c8fcf21f2)

## 5. Read the trace, not the code

This is the point of the lab. The question *"which chunks produced that sentence?"*
is now answerable without reading a line of the agent.

In [0]:
import time

mlflow.flush_trace_async_logging()
exp_id = mlflow.get_experiment_by_name(EXPERIMENT).experiment_id

df = None
for _ in range(10):
    df = mlflow.search_traces(locations=[exp_id], max_results=5)
    if len(df):
        break
    time.sleep(6)

print(f"  {len(df)} trace(s)\n")
tr = mlflow.get_trace(df.trace_id.iloc[0])
by_id = {s.span_id: s for s in tr.data.spans}

def depth(s):
    d = 0
    while s.parent_id and s.parent_id in by_id:
        s = by_id[s.parent_id]; d += 1
    return d

for s in tr.data.spans:
    dur = (s.end_time_ns - s.start_time_ns) / 1e9
    print(f"  {'  ' * depth(s)}{s.name:16} [{s.span_type}]  {dur:5.2f}s")

2026/09/30 09:21:52 INFO mlflow.utils.databricks_sql_warehouse: SQL warehouse 'c8729519c456cb8e' is STOPPED; starting it and waiting up to 1200s for RUNNING.


  5 trace(s)

  answer           [AGENT]  13.56s
    get_order        [TOOL]   1.26s
    search_policy    [RETRIEVER]   0.47s


### The retriever span holds the evidence

Everything you need to judge grounding is in one span.

In [0]:
ret = [s for s in tr.data.spans if str(s.span_type).endswith("RETRIEVER")][0]

ins = ret.inputs if isinstance(ret.inputs, dict) else json.loads(ret.inputs)
print(f"  query    : {ins.get('query')}")
print(f"  audience : {ins.get('audience', 'customer')}")
print(f"  k        : {ins.get('k', 3)}\n")

outs = ret.outputs if isinstance(ret.outputs, list) else json.loads(ret.outputs)
for h in outs:
    print(f"  {h['chunk_id']:12} {h['doc_id']}  score={h['score']:.4f}  {h['title'][:38]}")
    print(f"      {h['text'][:100]}...")

print("\n  If a claim in the answer is not in one of these chunks, the model invented it.")

  query    : delivery time and late delivery options
  audience : customer
  k        : 3

  DOC-003-C01  DOC-003  score=0.5984  Delivery timescales by region
      Express delivery is available to all regions for an additional 18 GBP and halves the stated timescal...
  DOC-003-C00  DOC-003  score=0.5677  Delivery timescales by region
      Standard delivery to the UK is 2 to 3 working days. Delivery to Benelux and DACH is 3 to 5 working d...
  DOC-005-C00  DOC-005  score=0.5521  Billing, invoices and payment terms
      Standard and plus tier customers are charged at the point of order. Premier tier customers may reque...

  If a claim in the answer is not in one of these chunks, the model invented it.


## 6. Your traces are Delta tables

This is what the Unity Catalog trace location bought you. Not a log file — **a
governed table you can query, join and aggregate.**

> ⚠️ **The schema is OpenTelemetry, not MLflow.** Columns are `name`, `trace_id`,
> `span_id`, `parent_span_id`, `start_time_unix_nano`, `end_time_unix_nano`,
> `attributes` (a `variant`) and `status` (a **struct** — so it is `status.code`,
> not `status_code`). Run `DESCRIBE` on the table before guessing; the first draft
> of this cell failed with `UNRESOLVED_COLUMN.WITH_SUGGESTION` for exactly that
> reason.

In [0]:
tables = [r["tableName"] for r in spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").collect()
          if r["tableName"].startswith(exp_id)]
print(f"  trace tables for experiment {exp_id}:")
for t in tables:
    print(f"    {t}")

spans_tbl = f"{CATALOG}.{SCHEMA}.{exp_id}_otel_spans"

print(f"\n  where does the time go, across every run so far?\n")
display(spark.sql(f"""
    SELECT
      name                                                               AS span,
      count(*)                                                           AS calls,
      round(avg((end_time_unix_nano - start_time_unix_nano)/1e9), 2)     AS avg_s,
      round(max((end_time_unix_nano - start_time_unix_nano)/1e9), 2)     AS max_s,
      sum(CASE WHEN status.code = 'STATUS_CODE_ERROR' THEN 1 ELSE 0 END) AS errors
    FROM {spans_tbl}
    GROUP BY name
    ORDER BY avg_s DESC
"""))

  trace tables for experiment 2155530130638727:
    2155530130638727_otel_annotations
    2155530130638727_otel_logs
    2155530130638727_otel_metrics
    2155530130638727_otel_spans
    2155530130638727_trace_metadata
    2155530130638727_trace_unified

  where does the time go, across every run so far?



span,calls,avg_s,max_s,errors
answer,7,18.64,21.1,0
get_order,8,4.61,9.6,0
search_policy,8,2.06,3.91,0


## 7. What you built

| Piece | Cell | Why it matters |
|---|---|---|
| `trace_location=UnityCatalog(...)` | 1 | without it, traces go to legacy storage **silently** |
| The assertion | 1 | a trace in the UI does not prove UC storage |
| `span_type="RETRIEVER"` | 2 | what makes Session 6's scorers able to read retrieval |
| The span tree | 5 | where the time actually went |
| The retriever span | 5 | the exact chunks, with scores — the grounding evidence |
| Trace tables in SQL | 6 | "how often does retrieval return nothing?" is a query |

### Try this before you move on

1. Ask something the documents do not cover — *"what is the refund approval limit?"*
   Then open the retriever span. **The chunks that came back are the reason it
   refused.** That is a correct refusal, and Lab 6A shows a judge marking it wrong.
2. Run the agent three times on the same question and compare the retriever spans.
   Identical? The model's *query* may not be.
3. Note the `audience: customer` input on the retriever span. That filter is the
   only thing keeping internal documents out — see [Lab 7B](#).